# Tutorial 3 — Normalization, Discretization & PCA on 500 Zanzibar Farms
**Z3003 Data Mining: Scaling to Large Data** · IIT Madras Zanzibar · Thursday 1 October 2026, 4:45–5:30 pm · Dr. Innocent Nyalala

### What you will do today
| Part | Time | Task | Lecture |
|---|---|---|---|
| A | 5 min | Load, clean and inspect the data (done for you) | Week 2 |
| B | 10 min | Normalize with min-max and z-score; check one value **by hand**; avoid leakage | Lecture 7 |
| C | 12 min | Discretize: equal-frequency bins, then **entropy and information gain** to find the best split | Lecture 7 |
| D | 10 min | PCA: standardize, scree plot, loadings, 2-D picture of all 500 farms | Lecture 8 |
| E | 8 min | Graded exercise (4 questions) | — |

**How to work:** read each explanation cell first, then run or complete the code cell under it. Every task ends with a ✅ **Checkpoint** — the number you should see. If yours differs, stop and fix it before moving on.
Lines starting `# TODO` are yours to complete. Cells starting `✍️` need a written answer in your own words.

## Part A — Load and clean the data (done for you)
Our dataset has **500 farms** and 9 columns. **24 farms have a missing `yield_tonnes`.** In Week 2 we learned that deleting them wastes data, so we fill each missing yield with the **median yield of the same crop** (a crop-aware imputation). After this, all 500 farms are complete.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

df = pd.read_csv("datasets/W1_zanzibar_crop_sample.csv")
print("Missing yields before:", df["yield_tonnes"].isna().sum())
df["yield_tonnes"] = df.groupby("crop")["yield_tonnes"].transform(lambda s: s.fillna(s.median()))
print("Missing yields after: ", df["yield_tonnes"].isna().sum())

num_cols = ["farm_size_ha", "yield_tonnes", "rainfall_mm", "price_per_kg", "revenue_usd"]
print(df.shape)
df[num_cols].describe().round(2)

✅ **Checkpoint A:** 24 missing → 0 missing; shape `(500, 9)`. Look at the `min` and `max` rows: farm size stays below 10, revenue goes above 24,000. **That difference in scale is the problem Part B fixes.**

## Part B — Normalization (Lecture 7)
**Why?** Distance-based methods (k-NN, k-means) and PCA add attributes together. An attribute measured in big numbers (revenue, rainfall) would drown out one measured in small numbers (price per kg).

| Method | Formula | In words |
|---|---|---|
| Min-max | $v' = \dfrac{v - \min}{\max - \min}$ | fraction of the way from the smallest to the largest value → range 0 … 1 |
| Z-score | $z = \dfrac{v - \mu}{\sigma}$ | how many standard deviations above (+) or below (−) the average |

**Symbols:** $v$ = one farm's value; $v'$, $z$ = the new value; $\min$, $\max$ = smallest / largest in the column; $\mu$ = column mean; $\sigma$ = column standard deviation.

### B1. Normalize all five numeric columns with scikit-learn
`fit` learns the statistics (min/max or μ/σ); `transform` applies them.

In [ ]:
mm = MinMaxScaler()
df_minmax = pd.DataFrame(mm.fit_transform(df[num_cols]), columns=num_cols)

zs = StandardScaler()
df_z = pd.DataFrame(zs.fit_transform(df[num_cols]), columns=num_cols)

print("Min-max: every column now runs from", df_minmax.min().min(), "to", df_minmax.max().max())
print("Z-score means (≈0):", df_z.mean().round(3).tolist())
print("Z-score std  (=1): ", df_z.std(ddof=0).round(3).tolist())

### B2. Check one value by hand — never trust a library blindly
Farm 1 has `rainfall_mm = 1780.8`. The column's min is 682.2 and max is 2016.8.
$$v' = \frac{1780.8 - 682.2}{2016.8 - 682.2} = \frac{1098.6}{1334.6} = 0.8232$$
Complete the code to compute this with pandas and compare it with scikit-learn's answer.

In [ ]:
v = df.loc[0, "rainfall_mm"]
vmin = None     # TODO: smallest rainfall in the column
vmax = None     # TODO: largest rainfall in the column
by_hand = None  # TODO: apply the min-max formula
print(f"by hand: {by_hand:.4f}   scikit-learn: {df_minmax.loc[0, 'rainfall_mm']:.4f}")

✅ **Checkpoint B2:** both print **0.8232**.

### B3. Avoid data leakage: fit on TRAIN only
When you later build a model, split the data first and learn μ and σ from the **training rows only** — then apply those same numbers to the test rows. Otherwise the test data 'leaks' into preprocessing and your accuracy looks better than it really is.

In [ ]:
X_train, X_test = train_test_split(df[num_cols], test_size=0.3, random_state=42)
scaler = StandardScaler()
# TODO: fit the scaler on X_train ONLY
# TODO: transform X_train and X_test with the SAME fitted scaler (store as X_train_z, X_test_z)
print("rainfall mean learned from train:", round(scaler.mean_[2], 1), "  full-data mean:", round(df['rainfall_mm'].mean(), 1))

✅ **Checkpoint B3:** train mean ≈ **1409.8** vs full-data mean **1397.9** — slightly different, because the test rows were (correctly) kept out.

## Part C — Discretization with entropy (Lecture 7)
**Goal:** turn a number into categories **using the class label to choose the boundary.**
Our class label: is a farm's yield **High** or **Low**? We split at the median yield (2.19 t), which gives two nearly equal groups.

**Entropy** measures how mixed a group is: $H(S) = -\sum_i p_i \log_2 p_i$ — 0 for a pure group, 1 for a 50/50 mix.
**Information gain** of a split: $IG = H(\text{before}) - \left[\frac{n_L}{n}H(L) + \frac{n_R}{n}H(R)\right]$ — how much mixing the split removes.

### C1. Unsupervised first: three equal-frequency rainfall bins

In [ ]:
# TODO: use pd.qcut to cut rainfall_mm into 3 equal-frequency bins labelled Low / Medium / High
df["rain_bin"] = None  # TODO: replace None with pd.qcut(...)
print(df["rain_bin"].value_counts().sort_index())

✅ **Checkpoint C1:** 167 / 166 / 167 farms. Boundaries ≈ 1279 mm and 1503 mm. Notice: these cuts never looked at yield.

### C2. Write your own entropy function

In [ ]:
df["yield_class"] = np.where(df["yield_tonnes"] > df["yield_tonnes"].median(), "High", "Low")

def entropy(labels):
    # TODO: proportions p_i of each class, then return -sum(p * log2(p))
    pass

print("Entropy of all 500 farms:", round(entropy(df["yield_class"]), 4))
print("Check a pure group (should be 0):", entropy(pd.Series(["High"] * 5)))

✅ **Checkpoint C2:** **1.0** (247 High vs 253 Low is almost exactly 50/50) and **0** for the pure group. (A tiny −0.0 is fine.)

### C3. Information gain — and a discovery
Write `info_gain(x, labels, split)`, then search **every** midpoint between neighbouring values of an attribute for the best split. Do it for **rainfall** and for **price per kg**.

In [ ]:
def info_gain(x, labels, split):
    left = x < split
    # TODO: weighted entropy after the split, subtracted from entropy before
    pass

def best_split(col):
    x = df[col].values; y = df["yield_class"].values
    values = np.sort(np.unique(x))
    mids = (values[:-1] + values[1:]) / 2
    gains = [info_gain(x, y, m) for m in mids]
    i = int(np.argmax(gains))
    return mids[i], gains[i]

for col in ["rainfall_mm", "price_per_kg"]:
    s, g = best_split(col)
    print(f"{col:13s} best split = {s:8.3f}   information gain = {g:.4f}")

✅ **Checkpoint C3:** rainfall's best gain is only ≈ **0.005** (at ≈ 1029 mm); price's best gain is ≈ **0.39** at ≈ **$1.20/kg**.

✍️ **C4. In 2–3 sentences:** what does this tell you about which attribute is informative for yield class in this dataset? Why might cheap crops (below $1.20/kg — cassava, coconut, rice) have *higher* tonnage?

*Your answer:*

## Part D — PCA (Lecture 8)
**Goal:** replace 5 correlated attributes by a few uncorrelated **principal components** that keep most of the variance.
**Rule from Lecture 8:** standardize first, otherwise the attribute with the biggest numbers wins.

### D1. PCA on the standardized data

In [ ]:
pca = PCA()           # keep all 5 components for now
# TODO: fit pca on df_z (the z-scored data from Part B)
ratio = pca.explained_variance_ratio_
print("variance explained:", ratio.round(3))
print("cumulative:        ", ratio.cumsum().round(3))
print("eigenvalues:       ", pca.explained_variance_.round(2))

✅ **Checkpoint D1:** ≈ **0.423, 0.204, 0.198, 0.164, 0.011**; eigenvalues ≈ 2.12, 1.02, 0.99, 0.82, 0.06.
*(scikit-learn divides by n − 1, so eigenvalues differ from the lecture's by a factor 500/499 — the percentages are identical.)*

### D2. Scree plot

In [ ]:
# TODO: bar chart of ratio*100 for PC1..PC5, plus a line for the cumulative %


✍️ **D3.** Using the Kaiser rule (keep eigenvalues > 1) and the cumulative %, how many components would you keep? Justify in one sentence.

*Your answer:*

### D4. What do the components mean? Loadings
`pca.components_[0]` holds the weight of each attribute in PC1.

In [ ]:
loadings = pd.DataFrame(pca.components_[:2], columns=num_cols, index=["PC1", "PC2"]).round(3)
loadings
# ✍️ Name PC1 and PC2 in plain words based on their largest weights.

### D5. All 500 farms in two dimensions

In [ ]:
scores = PCA(n_components=2).fit_transform(df_z)
# TODO: scatter scores[:,0] vs scores[:,1], one colour per crop, with a legend


✅ **Checkpoint D5:** clove and cashew farms sit apart from the others along PC1 — PCA never saw the crop label.

## Part E — Graded exercise (8 min)
**E1.** Min-max normalize `price_per_kg` alone and verify farm 1's value by hand (farm 1 price = 0.26; column min 0.20, max 9.06).

In [ ]:
# E1: your code


**E2.** Cut `farm_size_ha` into 4 **equal-width** bins with `pd.cut` and report the counts. Why are they so unequal?

In [ ]:
# E2: your code


**E3.** Run PCA on `df[num_cols]` **without** standardizing. Print the variance-explained ratios and compare with D1.

In [ ]:
# E3: your code


✍️ **E4.** In 2–3 sentences, explain why PCA must be run on standardized data here. **Quote your own E3 numbers.**

*Your answer:*

---
### Submission
Save with all outputs visible as **`YourName_W3_Tutorial3.ipynb`** and upload to the course portal before **5:30 pm**.

### Mini-project
The Mini-Project is announced today — read **`Mini_Project_Brief.pdf`**. Parts B–D of this notebook are exactly the *Preprocessing* section of your project.